# TuniShop — Machine learning 04. Prédiction de la demande par produit

**Auteure :** Sarra Abdeljaouad · **Phase :** machine learning · **Source :** TuniShop_DWH (vues `bi`), via `05_Data/features` · **Découpage :** apprentissage 2023-2024, test 2025

## Objectif

Prévoir la **quantité vendue de chaque produit, chaque mois de 2025**, depuis le 31/12/2024 (horizon 1 à 12 mois). C'est l'outil qui manque à l'approvisionnement : l'EDA a montré que les ruptures arrivent aux pics de vente.

| Modèle | Rôle | Idée |
|---|---|---|
| Naïf saisonnier | référence | même mois de l'année précédente (à défaut : moyenne du produit, puis de sa catégorie) |
| Moyenne de l'année précédente | **référence** | moyenne mensuelle du produit l'an dernier |
| Gradient boosting (perte de Poisson) | candidat | adaptée aux comptages (beaucoup de petites valeurs et de zéros) |
| Gradient boosting (perte quadratique) | candidat | perte classique |
| Catégorie × part produit (v2) | candidat | prévision par catégorie (saisonnalité et croissance de l'année précédente), répartie selon la part de chaque produit (`features.prevision_demande_v2`) |

**Mesures :** **WAPE** (somme des erreurs absolues / quantité totale réelle : l'erreur « en volume », robuste aux mois à zéro), MAE (erreur moyenne en unités) et biais. La MAPE n'a pas de sens ici (18 % des mois sont à zéro).

## 0. Préparation

In [1]:
# --- Bibliothèques
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# --- Module commun du projet (dossier 00_Commun)
sys.path.insert(0, "../00_Commun")
from tunishop_analytics import config as cfg, chargement, features, temporal_split, validation, publication

warnings.filterwarnings("ignore")
cfg.appliquer_style()
sql = chargement.requete                       # sql("SELECT ...") -> exécuté par SQL Server
pd.set_option("display.float_format", lambda v: f"{v:,.3f}".replace(",", " "))

def montrer(nom):
    """Enregistre la figure en cours dans 06_Exports/Figures/<nom>.png, puis l'affiche."""
    cfg.sauver_figure(plt.gcf(), nom)
    plt.show()

for dossier in (cfg.DOSSIER_MODELES, cfg.DOSSIER_PREDICTIONS):
    dossier.mkdir(parents=True, exist_ok=True)

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline

In [2]:
chargement.ouvrir_connexion()

demande = features.lire("ml_demande")
produits_mois = features.lire("produits_mois")          # historique mensuel (pour la version 2)
X_COLS, CIBLE = features.colonnes_ml("demande"), "quantite"
app, test = demande[demande["periode"] == "apprentissage"], demande[demande["periode"] == "test"]
print(f"apprentissage (mois de 2024) : {len(app):,} lignes ; test (mois de 2025) : {len(test):,} lignes ; "
      f"{test['produit_id'].nunique()} produits".replace(",", " "))

# 🔗 Contrôle SQL direct : quantité 2025 des lignes abouties
q_sql = chargement.valeur("""
    SELECT SUM(v.quantite) FROM bi.v_fact_ventes v
    JOIN bi.v_dim_statut_commande s ON s.statut_commande_sk = v.statut_commande_sk
    JOIN bi.v_dim_date d ON d.date_sk = v.date_commande_sk
    WHERE s.famille_statut = 'aboutie' AND v.produit_sk > 0 AND d.annee = 2025""")
print(f"Quantité 2025 — SQL : {int(q_sql):,} ; jeu de données : {int(test[CIBLE].sum()):,}".replace(",", " "))
test[CIBLE].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).to_frame("quantité mensuelle par produit (2025)").T

Connecté à TuniShop_DWH sur localhost (pilote : ODBC Driver 17 for SQL Server)
apprentissage (mois de 2024) : 6 223 lignes ; test (mois de 2025) : 6 992 lignes ; 595 produits
Quantité 2025 — SQL : 69 876 ; jeu de données : 69 876


,count,mean,std,min,25%,50%,75%,90%,99%,max
quantité mensuelle par produit (2025),6 992.000,9.994,24.918,0.000,1.000,4.000,10.000,24.000,84.090,1 037.000


## 1. Choix de la perte sur la validation

Validation interne à l'apprentissage : on apprend sur les mois de janvier à août 2024 et on mesure sur septembre à décembre 2024. Les deux références sont mesurées sur la même validation : un modèle de ML n'est retenu que s'il les bat.

In [3]:
candidats = {
    "Gradient boosting (Poisson)": HistGradientBoostingRegressor(loss="poisson", learning_rate=0.05, max_iter=400, max_leaf_nodes=31, random_state=0),
    "Gradient boosting (quadratique)": HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.05, max_iter=400, max_leaf_nodes=31, random_state=0),
}
a_fit, a_val = app[app["mois_num"] <= 8], app[app["mois_num"] > 8]
# Les références sont aussi mesurées sur la validation : si aucune n'est battue, on garde la référence
valid = [{"modele": "Naïf saisonnier", "WAPE validation": validation.metriques_regression(
              a_val[CIBLE], a_val["meme_mois_annee_prec"].fillna(a_val["moy_annee_prec"]).fillna(a_val["moy_categorie_annee_prec"]))["WAPE"]},
         {"modele": "Moyenne de l'année précédente (référence)", "WAPE validation": validation.metriques_regression(
              a_val[CIBLE], a_val["moy_annee_prec"].fillna(a_val["moy_categorie_annee_prec"]))["WAPE"]}]
for nom, m in candidats.items():
    pipe = make_pipeline(features.preprocesseur("demande", echelle=False), m).fit(a_fit[X_COLS], a_fit[CIBLE])
    valid.append({"modele": nom, "WAPE validation": validation.metriques_regression(a_val[CIBLE], pipe.predict(a_val[X_COLS]))["WAPE"]})
# Version 2 : pas d'apprentissage, elle n'utilise que l'année précédente (aucune fuite)
V2 = "Catégorie × part produit (v2)"
valid.append({"modele": V2, "WAPE validation": validation.metriques_regression(
    a_val[CIBLE], features.prevision_demande_v2(a_val, produits_mois))["WAPE"]})
valid = pd.DataFrame(valid)
cfg.exporter_table(valid, "ML04_validation", index=False)
valid

Publication : connecté à TuniShop_ANALYTICS sur localhost
  ✔ ana.ml04_validation                               5 lignes


,modele,WAPE validation
0,Naïf saisonnier,0.607
1,Moyenne de l'année précédente (référence),0.571
2,Gradient boosting (Poisson),0.571
3,Gradient boosting (quadratique),0.602
4,Catégorie × part produit (v2),0.551


## 2. Résultats sur 2025 (test)

In [4]:
previsions = {
    "Naïf saisonnier": test["meme_mois_annee_prec"].fillna(test["moy_annee_prec"]).fillna(test["moy_categorie_annee_prec"]).values,
    "Moyenne de l'année précédente (référence)": test["moy_annee_prec"].fillna(test["moy_categorie_annee_prec"]).values,
}
finals = {}
for nom, m in candidats.items():
    finals[nom] = make_pipeline(features.preprocesseur("demande", echelle=False), m).fit(app[X_COLS], app[CIBLE])
    previsions[nom] = np.clip(finals[nom].predict(test[X_COLS]), 0, None)
previsions[V2] = features.prevision_demande_v2(test, produits_mois).values

lignes = []
for nom, p in previsions.items():
    m = validation.metriques_regression(test[CIBLE], p)
    agr = test.assign(p=p).groupby(["categorie", "mois"])[[CIBLE, "p"]].sum()
    lignes.append({"cas": "demande", "modele": nom, "role": "candidat" if (nom in candidats or nom == V2) else "référence",
                   "WAPE_validation": valid.set_index("modele")["WAPE validation"].get(nom, np.nan),
                   "WAPE_produit_mois": m["WAPE"], "MAE_produit_mois": m["MAE"], "biais": m["biais"],
                   "WAPE_categorie_mois": validation.metriques_regression(agr[CIBLE], agr["p"])["WAPE"]})
resultats = pd.DataFrame(lignes)
reference = resultats.loc[resultats["modele"].str.contains("référence"), "WAPE_produit_mois"].iloc[0]
resultats["gain_vs_reference"] = 1 - resultats["WAPE_produit_mois"] / reference
# Règle de choix (validation) : un candidat n'est retenu que s'il bat la référence d'au moins 1 %.
# À performance égale, on garde le modèle le plus simple.
v = valid.set_index("modele")["WAPE validation"]
meilleur_ml = v[list(candidats) + [V2]].idxmin()
REFERENCE = "Moyenne de l'année précédente (référence)"
RETENU = meilleur_ml if v[meilleur_ml] < 0.99 * v[REFERENCE] else REFERENCE
print(f"Validation : meilleur candidat = {meilleur_ml} ({v[meilleur_ml]:.4f}), référence = {v[REFERENCE]:.4f} -> retenu : {RETENU}")
resultats["retenu"] = resultats["modele"] == RETENU
cfg.exporter_table(resultats, "ML_metriques_demande", index=False)
resultats

Validation : meilleur candidat = Catégorie × part produit (v2) (0.5514), référence = 0.5712 -> retenu : Catégorie × part produit (v2)
  ✔ ana.ml_metriques_demande                          5 lignes


,cas,modele,role,WAPE_validation,WAPE_produit_mois,MAE_produit_mois,biais,WAPE_categorie_mois,gain_vs_reference,retenu
0,demande,Naïf saisonnier,référence,0.607,0.520,5.197,-0.209,0.246,-0.089,False
1,demande,Moyenne de l'année précédente (référence),référence,0.571,0.477,4.771,-0.209,0.270,0.000,False
2,demande,Gradient boosting (Poisson),candidat,0.571,0.486,4.856,-0.162,0.236,-0.018,False
3,demande,Gradient boosting (quadratique),candidat,0.602,0.534,5.336,-0.093,0.169,-0.119,False
4,demande,Catégorie × part produit (v2),candidat,0.551,0.475,4.751,-0.235,0.274,0.004,True


> **Lecture.** Sur la validation (septembre à décembre 2024), la **version 2** (prévision par catégorie puis répartition par produit) obtient une WAPE de **55,1 %** contre 57,1 % pour la moyenne de l'année précédente, soit 3,5 % de mieux : selon la règle fixée à l'avance (au moins 1 % de mieux), elle est **retenue**. Le meilleur gradient boosting (Poisson, 57,1 %) ne bat pas la référence. Sur le test 2025, la version 2 reste devant mais de peu : WAPE de **47,5 %** contre 47,7 % (+0,4 %). Au niveau **catégorie × mois**, les modèles de ML restent les meilleurs (16,9 % pour le boosting quadratique). Tous les modèles **sous-estiment** 2025 (biais de -9 à -24 %) : la croissance de 2025 est plus forte que celle observée fin 2024.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
tot = test.assign(**{k: v for k, v in previsions.items()}).groupby("mois")[[CIBLE] + list(previsions)].sum()
axes[0].plot(tot.index, tot[CIBLE], color="black", lw=2.5, marker="o", label="réel 2025")
for nom, coul in [("Moyenne de l'année précédente (référence)", cfg.ORANGE), (meilleur_ml, cfg.BLEU)] if meilleur_ml != REFERENCE else [(REFERENCE, cfg.ORANGE)]:
    axes[0].plot(tot.index, tot[nom], marker=".", color=coul, label=nom)
axes[0].set_title("Quantité totale vendue par mois en 2025"); cfg.format_milliers(axes[0]); axes[0].legend(fontsize=8)
par_cat = pd.DataFrame({nom: test.assign(p=previsions[nom]).groupby("categorie").apply(
    lambda g: validation.metriques_regression(g[CIBLE], g["p"])["WAPE"]) for nom in ["Moyenne de l'année précédente (référence)", meilleur_ml]})
par_cat.sort_values(meilleur_ml).plot.barh(ax=axes[1], color=[cfg.ORANGE, cfg.BLEU])
axes[1].set_title("WAPE par catégorie (produit x mois)"); cfg.format_pourcent(axes[1], "x")
axes[1].set_ylabel(""); axes[1].legend(fontsize=7, loc="lower right")
montrer("ML04_1_demande_2025")
cfg.exporter_table(par_cat, "ML04_wape_categories")

> **Lecture.** La référence donne un niveau moyen **sans saisonnalité** (courbe plate) ; la version 2 reprend la **saisonnalité de chaque catégorie** (pics de mars, de la rentrée et de novembre) mais, comme la référence, **sous-estime le niveau** de 2025. Les écarts par catégorie sont faibles et changent de sens selon la catégorie.

In [6]:
# Erreur selon le volume du produit : les petits produits sont-ils les plus difficiles ?
volume = test.groupby("produit_id")[CIBLE].sum()
classes = pd.qcut(volume.rank(method="first"), 4, labels=["Q1 (petits volumes)", "Q2", "Q3", "Q4 (gros volumes)"])
t = test.assign(p=previsions[meilleur_ml], classe=test["produit_id"].map(classes))
erreur_volume = t.groupby("classe", observed=True).apply(lambda g: pd.Series({
    "produits": g["produit_id"].nunique(), "part_du_volume": g[CIBLE].sum() / t[CIBLE].sum(),
    "WAPE": validation.metriques_regression(g[CIBLE], g["p"])["WAPE"]}))
cfg.exporter_table(erreur_volume, "ML04_erreur_par_volume")
erreur_volume

  ✔ ana.ml04_erreur_par_volume                        4 lignes


,produits,part_du_volume,WAPE
classe,,,
Q1 (petits volumes),149.000,0.016,1.753
Q2,149.000,0.069,0.648
Q3,148.000,0.168,0.524
Q4 (gros volumes),149.000,0.747,0.420


> **Lecture.** L'erreur dépend surtout du volume du produit : **175 %** pour le quart des produits les moins vendus (1,6 % du volume), **42 %** pour le quart des plus vendus (75 % du volume). Prévoir mois par mois un produit vendu 2 ou 3 fois par mois est presque impossible ; l'enjeu est la précision sur les gros volumes.

## 3. Sauvegarde

In [7]:
# On sauvegarde le meilleur gradient boosting (point de départ des améliorations) et le nom du modèle retenu.
# La version 2 n'a pas de paramètre appris : elle se recalcule avec features.prevision_demande_v2.
meilleur_gb = v[list(candidats)].idxmin()
joblib.dump({"pipeline": finals[meilleur_gb], "variables": X_COLS, "appris_sur": "mois de 2024 (historique 2023)",
             "retenu_en_production": RETENU,
             "methode_v2": "features.prevision_demande_v2(lignes, produits_mois)"}, cfg.DOSSIER_MODELES / "demande_modele.joblib")
pred = test[["produit_id", "mois", "categorie", CIBLE]].copy()
for nom, p in previsions.items():
    pred[nom] = p
pred["prevision_retenue"] = previsions[RETENU]          # lue par Power BI (pbi.v_ml_demande), quel que soit le modèle retenu
cfg.exporter_resultat(pred, cfg.DOSSIER_PREDICTIONS / "demande_2025_test.parquet")
print("Modèle : 06_Exports/Modeles/demande_modele.joblib ; prédictions : 06_Exports/Predictions/demande_2025_test.parquet")

  ✔ ana.ml_pred_demande_2025                      6 992 lignes
Modèle : 06_Exports/Modeles/demande_modele.joblib ; prédictions : 06_Exports/Predictions/demande_2025_test.parquet


## 4. Synthèse

> **Lecture.** **Modèle retenu : version 2, « catégorie × part produit »** (WAPE 2025 de 47,5 % contre 47,7 % pour la référence ; 3,5 % de mieux en validation). Elle n'a aucun paramètre appris : elle se recalcule chaque mois avec `features.prevision_demande_v2` à partir de l'année précédente. Le gain sur 2025 est faible : la prochaine piste est la **correction de la croissance** (biais de -23,5 %), par exemple avec la tendance de la prévision des ventes par catégorie. Le meilleur gradient boosting reste enregistré comme point de départ (`06_Exports/Modeles/demande_modele.joblib`).

In [8]:
chargement.fermer_connexion()

Connexion fermée.
